# 03 – Inputs of the two runs: fetal and postnatal

**Question.** Which cells and which L–R links go into each CausalCCC run?

Rules (from thymus case 2, notebook 03, with the changes needed for two comparable runs):
- **same cell numbers in both runs**, drawn at random (fixed seed) within each population and age class:
  CD8αα(I) 1,000; DC1 300 (postnatal has only 312); aDC1 all its cells (fetal 100, postnatal 35);
- **same L–R links in both runs**: the claimed pairs (XCL1 → XCR1, XCL2 → XCR1) always, then the candidates of
  notebook 02 ranked by S5 mean, up to 25 S5 pairs, with the same clean-up rules as case 2 (pairs whose gene links are
  already in are skipped; IDE as "receptor" and LCK as "ligand" are excluded);
- **metadata**: senders `donor`, `method`, `n_counts`; receivers `population`, `donor`, `method`, `n_counts`.
  `population` is dropped on the sender side (one population only) and `stage` is dropped (each donor has one age, so
  stage was redundant with donor in case 2, and age class is constant within a run).
- **sequencing depth** (`n_counts`, total raw counts per cell, computed by the authors) is a network variable, as in
  Minh's runs A and B (`notebooks/thymus_early/`, notebooks 05 and 08): without it, housekeeping genes whose counts
  follow the depth of the cell form a module of links between themselves. The hubs of the case 2 network (GAPDH,
  ACTB, SERF2, PPIA, EEF1A1, TPT1, PTMA) look like that module.

In [1]:
from pathlib import Path
import gzip
import numpy as np
import pandas as pd
import anndata as ad
import scipy.io

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 60)

ROOT = Path("../..")
THY = ROOT / "data/thymus"
RES = ROOT / "results/thymus_case3_xcl1_age"
OUT = RES / "03_run_inputs"
OUT.mkdir(parents=True, exist_ok=True)

AGES = ["fetal", "postnatal"]
CAP = {"CD8αα(I)": 1000, "DC1": 300, "aDC1": None}      # None: all cells
SEED = 0
N_LINKS = 25
METADATA = ["population", "donor", "method", "n_counts"]   # columns written to cells.csv; notebook 04 picks them per side
ACTIVATION_GENES = ["CCR7", "LAMP3", "CD40", "CD80", "CD86", "IL12B"]
MAX_PIVOTS = 30

## 1. Cells of each run

In [2]:
cells = pd.read_csv(RES / "01_cell_populations/selected_cells.csv", index_col=0)

def draw(d):
    cap = CAP[d.population.iloc[0]]
    return d if cap is None or len(d) <= cap else d.sample(n=cap, random_state=SEED)

runs = {age: pd.concat([draw(d) for _, d in cells[cells.age_class == age].groupby("population")]) for age in AGES}
pd.DataFrame({age: r.population.value_counts() for age, r in runs.items()}).reindex(["CD8αα(I)", "DC1", "aDC1"])

,fetal,postnatal
population,,
CD8αα(I),1000,1000
DC1,300,300
aDC1,100,35


In [3]:
# Donors and chemistry per run
pd.concat({age: r.groupby("population").agg(donors=("donor", "nunique"),
                                             share_5GEX=("method", lambda m: (m == "5GEX").mean()))
           for age, r in runs.items()}, axis=1).round(2)

fetal            postnatal           
           donors share_5GEX    donors share_5GEX
population                                       
CD8αα(I)       14       0.68         8       0.60
DC1            15       0.35         8       0.67
aDC1           11       0.63         6       0.91

**Reading.**
- **Cell numbers:** the fetal run has 1,400 cells (1,000 CD8αα(I), 300 DC1, 100 aDC1) and the postnatal run 1,335
  (1,000, 300, 35). Only aDC1 differs, because postnatal aDC1 has no more cells.
- **Donors:** the senders come from 14 fetal and 8 postnatal donors.
- **Chemistry:** the share of 5′ cells differs between runs for the receivers (DC1: 35% fetal, 67% postnatal; aDC1:
  63% and 91%). `method` is a network variable in both runs.

## 2. The L–R links, shared by both runs

In [4]:
DB = THY / "cellphonedb"
gene = pd.read_csv(DB / "gene_input.csv").drop_duplicates("uniprot").set_index("uniprot")
gene["gene_name"] = gene.gene_name.replace({"NECTIN1": "PVRL1", "NECTIN2": "PVRL2", "NECTIN3": "PVRL3"})
cplx = pd.read_csv(DB / "complex_input.csv").set_index("complex_name")

def genes_of(name):
    if name in cplx.index:
        units = cplx.loc[name, ["uniprot_1", "uniprot_2", "uniprot_3", "uniprot_4"]].dropna()
        return [gene.gene_name[u] for u in units]
    return [name]

cand = pd.read_csv(RES / "02_ligand_receptor_candidates/candidates_cd8aa_to_dc1_by_age.csv")
CLAIMED = ["XCL1_XCR1", "XCL2_XCR1"]
EXCLUDED_RECEPTORS = {"IDE": "receptor is an enzyme (IDE)"}
EXCLUDED_LIGANDS = {"LCK": "ligand is an intracellular kinase (LCK)"}
assert set(CLAIMED) <= set(cand.pair)

cand["claimed"] = cand.pair.isin(CLAIMED)
ranked = cand.sort_values(["claimed", "s5_max_mean"], ascending=[False, False]).reset_index(drop=True)

chosen, skipped, gene_links = [], [], set()
for _, row in ranked.iterrows():
    if len(chosen) == N_LINKS:
        break
    lig, rec = genes_of(row.ligand), genes_of(row.receptor)
    new_links = {(l, r) for l in lig for r in rec}
    if set(rec) & set(EXCLUDED_RECEPTORS):
        skipped.append((row.pair, EXCLUDED_RECEPTORS[(set(rec) & set(EXCLUDED_RECEPTORS)).pop()]))
    elif set(lig) & set(EXCLUDED_LIGANDS):
        skipped.append((row.pair, EXCLUDED_LIGANDS[(set(lig) & set(EXCLUDED_LIGANDS)).pop()]))
    elif new_links <= gene_links:
        skipped.append((row.pair, "all gene links already included"))
    else:
        chosen.append(row.pair)
        gene_links |= new_links

links = ranked[ranked.pair.isin(chosen)].copy()
cols = ["pair", "ligand", "receptor", "orientation_rule", "claimed", "passes_fetal", "passes_postnatal",
        "ligand_frac_fetal", "ligand_frac_postnatal", "receptor_pops", "s5_max_mean"]
print("skipped while filling the list:", skipped)
links[cols].round(2)

skipped while filling the list: [('CD8 receptor_LCK', 'ligand is an intracellular kinase (LCK)')]


,pair,ligand,receptor,orientation_rule,claimed,passes_fetal,passes_postnatal,ligand_frac_fetal,ligand_frac_postnatal,receptor_pops,s5_max_mean
0,XCL1_XCR1,XCL1,XCR1,receptor flag,True,True,True,0.29,0.12,"DC1 | fetal, DC1 | postnatal",1.12
1,XCL2_XCR1,XCL2,XCR1,receptor flag,True,True,False,0.16,0.05,"DC1 | fetal, DC1 | postnatal",0.81
2,CD74_MIF,MIF,CD74,receptor flag,False,True,True,0.87,0.62,"DC1 | fetal, DC1 | postnatal, aDC1 | fetal, aD...",3.86
3,APP_CD74,APP,CD74,receptor flag,False,True,True,0.29,0.18,"DC1 | fetal, DC1 | postnatal, aDC1 | fetal, aD...",2.90
4,MIF_TNFRSF14,MIF,TNFRSF14,receptor flag,False,True,True,0.87,0.62,"DC1 | fetal, DC1 | postnatal, aDC1 | fetal, aD...",1.42
5,LGALS9_CD44,LGALS9,CD44,receptor flag,False,True,True,0.15,0.11,"DC1 | fetal, DC1 | postnatal, aDC1 | fetal, aD...",1.10
6,CD28_CD86,CD28,CD86,receptor flag,False,True,True,0.71,0.43,"DC1 | fetal, DC1 | postnatal, aDC1 | fetal, aD...",0.84
7,CD40LG_CD40,CD40LG,CD40,receptor flag,False,True,False,0.20,0.06,"DC1 | fetal, DC1 | postnatal, aDC1 | fetal, aD...",0.81
8,CD80_CD28,CD28,CD80,receptor flag,False,True,True,0.71,0.43,"aDC1 | fetal, aDC1 | postnatal",0.75
10,SPN_ICAM1,SPN,ICAM1,receptor flag,False,True,True,0.52,0.51,"DC1 | fetal, DC1 | postnatal, aDC1 | fetal, aD...",0.65


In [5]:
edges = pd.DataFrame([dict(ligands=l, receptors=r, pair=row.pair)
                      for _, row in links.iterrows()
                      for l in genes_of(row.ligand) for r in genes_of(row.receptor)]).drop_duplicates(["ligands", "receptors"])
ligand_genes, receptor_genes = sorted(edges.ligands.unique()), sorted(edges.receptors.unique())
receiver_pivots = sorted(set(receptor_genes) | set(ACTIVATION_GENES))
print(len(links), "L-R pairs ->", len(edges), "gene-gene links")
print("ligand genes:  ", len(ligand_genes), ligand_genes)
print("receptor genes:", len(receptor_genes), receptor_genes)
print("receiver pivots (receptors + activation genes):", len(receiver_pivots), "| limit", MAX_PIVOTS)
assert len(ligand_genes) <= MAX_PIVOTS and len(receiver_pivots) <= MAX_PIVOTS
print("on both sides:", sorted(set(ligand_genes) & set(receiver_pivots)))

25 L-R pairs -> 27 gene-gene links
ligand genes:   22 ['APP', 'CD200', 'CD28', 'CD38', 'CD40LG', 'CD6', 'CRTAM', 'FLT3LG', 'ICAM3', 'ITGA4', 'ITGAL', 'ITGB1', 'ITGB2', 'LGALS9', 'LTA', 'LTB', 'MIF', 'PDCD1', 'SIRPG', 'SPN', 'XCL1', 'XCL2']
receptor genes: 21 ['ALCAM', 'CADM1', 'CD200R1', 'CD274', 'CD40', 'CD44', 'CD47', 'CD74', 'CD80', 'CD86', 'FLT3', 'ICAM1', 'ITGAL', 'ITGB2', 'LTBR', 'PDCD1LG2', 'PECAM1', 'PLAUR', 'TNFRSF14', 'TNFRSF1B', 'XCR1']
receiver pivots (receptors + activation genes): 24 | limit 30
on both sides: ['ITGAL', 'ITGB2']


**Reading.**
- **The list:** 25 S5 pairs = XCL1 → XCR1 and XCL2 → XCR1 + 23 by S5 mean, giving 27 gene–gene links.
  `CD8 receptor_LCK` was skipped (LCK is an intracellular kinase).
- **Overlap with case 2:** 19 of the 25 pairs are the same as in case 2.
  - Dropped: the 6 pairs carried mostly by NK cells or by CD47 / SIRPA (IFNG, KLRF1, TIGIT, SELL, TNFSF14, CD47 →
    SIRPA).
  - Added: 6 pairs, PDCD1 → PDCD1LG2, CD6 → ALCAM, LTA → TNFRSF1B, LTA → TNFRSF14, FLT3LG → FLT3 and SIRPG → CD47.
- **Pivots:** 22 ligand genes; 21 receptor genes plus CCR7, LAMP3 and IL12B, so 24 receiver pivots, within the
  limit of 30.
- **Genes on both sides:** ITGAL and ITGB2 (LFA-1) are on both sides, as in case 2.

## 3. Export, one folder per run

`03_run_inputs/<run>/`: `counts.mtx.gz` (raw counts, genes × cells, all genes), `genes.csv`, `cells.csv` (group and
metadata), `interact_edges.tsv` (the same in both runs); `selected_links.csv` is written once.

In [6]:
raw = ad.read_h5ad(THY / "HTA07.A01.v02.entire_data_raw_count.h5ad", backed="r")
for age, r in runs.items():
    out = OUT / age
    out.mkdir(exist_ok=True)
    rows = np.sort(raw.obs_names.get_indexer(r.index))
    assert (rows >= 0).all()
    sub = raw[rows].to_memory()
    sub.var_names_make_unique()
    assert set(ligand_genes + receiver_pivots) <= set(sub.var_names)
    with gzip.open(out / "counts.mtx.gz", "wb") as f:
        scipy.io.mmwrite(f, sub.X.T.tocoo().astype(np.int32), field="integer")
    pd.Series(sub.var_names, name="gene").to_csv(out / "genes.csv", index=False)
    r.loc[sub.obs_names, ["group"] + METADATA + ["Age", "age_class"]].rename_axis("cell").to_csv(out / "cells.csv")
    edges[["ligands", "receptors"]].to_csv(out / "interact_edges.tsv", sep="\t", index=False)
    print(age, sub.shape, "->", sorted(p.name for p in out.iterdir()))
links[cols].to_csv(OUT / "selected_links.csv", index=False)

/tmp/ipykernel_415/112004171.py:1: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  raw = ad.read_h5ad(THY / "HTA07.A01.v02.entire_data_raw_count.h5ad", backed="r")


/tmp/ipykernel_415/112004171.py:7: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  sub = raw[rows].to_memory()


fetal (1400, 33694) -> ['cells.csv', 'counts.mtx.gz', 'genes.csv', 'interact_edges.tsv']


/tmp/ipykernel_415/112004171.py:7: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  sub = raw[rows].to_memory()


postnatal (1335, 33694) -> ['cells.csv', 'counts.mtx.gz', 'genes.csv', 'interact_edges.tsv']


## Summary

| | Fetal run | Postnatal run |
|---|---|---|
| Senders (`cd8aa`) | CD8αα(I) 1,000 (14 donors) | CD8αα(I) 1,000 (8 donors) |
| Receivers (`dc1`) | DC1 300 · aDC1 100 | DC1 300 · aDC1 35 |
| L–R links | 25 S5 pairs → 27 gene–gene links, the same in both runs | same |
| Metadata | senders: donor, method, n_counts · receivers: population, donor, method, n_counts | same |
| Inputs | `results/thymus_case3_xcl1_age/03_run_inputs/fetal/` | `.../postnatal/` |

Next: notebook 04 (`04_causalccc.qmd`), rendered once per run (see `run_both.sh`).